# 🎬 IMDB Sentiment Analysis — RNN Model Training
**Project:** Binary Sentiment Classification (Positive / Negative)  
**Model:** Recurrent Neural Network (RNN) using PyTorch  
**Dataset:** IMDB Movie Reviews (50,000 reviews)  

---
## 📋 Pipeline Overview
1. Install & Import Libraries
2. Load Dataset
3. Text Preprocessing (Clean → Tokenize → Stem)
4. Feature Extraction (TF-IDF Vectorization)
5. Train/Test Split
6. Build RNN Model
7. Train the Model
8. Evaluate Accuracy
9. Save Model + Vectorizer

## ✅ Step 1 — Install Dependencies
> Run this cell first (only needed once in Colab)

In [ ]:
# Install required packages (Colab already has most, but just in case)
!pip install torch torchvision --quiet
!pip install scikit-learn nltk pandas numpy --quiet

print('✅ All packages ready!')

## ✅ Step 2 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import re
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Libraries imported successfully!')
print(f'🖥️  Using device: {device}')

## ✅ Step 3 — Load Dataset
> **Colab Users:** Upload `IMDB Dataset.csv` to `/content/` before running this cell  
> Or mount Google Drive if it's stored there

In [ ]:
# ─── OPTION A: Load from Colab /content/ (after uploading manually) ───
DATASET_PATH = "/content/IMDB Dataset.csv"

# ─── OPTION B: Mount Google Drive (uncomment if stored on Drive) ───
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_PATH = "/content/drive/MyDrive/IMDB Dataset.csv"

df = pd.read_csv(DATASET_PATH, engine='python', on_bad_lines='skip')

print(f'📊 Dataset Shape: {df.shape}')
print(f'📋 Columns: {list(df.columns)}')
print(f'\n🏷️  Sentiment Distribution:')
print(df['sentiment'].value_counts())
df.head(3)

## ✅ Step 4 — Text Preprocessing
Cleaning pipeline:
- Lowercase → Remove HTML tags → Remove URLs → Remove punctuation
- Tokenize → Remove stop words → Stemming

In [ ]:
# Remove duplicates first
before = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'🗑️  Removed {before - len(df)} duplicate rows. Remaining: {len(df)}')

# Initialize preprocessing tools
stop_words = set(stopwords.words('english'))
ps = PorterStemmer()

def clean_text(text):
    """Full text cleaning pipeline"""
    text = text.lower()                          # Step 1: Lowercase
    text = re.sub(r'<.*?>', '', text)            # Step 2: Remove HTML tags
    text = re.sub(r'http\S+', '', text)          # Step 3: Remove URLs
    text = re.sub(r'[^a-z0-9\s]', '', text)     # Step 4: Remove punctuation
    tokens = word_tokenize(text)                 # Step 5: Tokenize
    cleaned = [ps.stem(w) for w in tokens       # Step 6: Stem + remove stopwords
               if w not in stop_words]
    return ' '.join(cleaned)

print('⏳ Cleaning text... (may take 1-2 minutes)')
df['review'] = df['review'].apply(clean_text)
print('✅ Text cleaning complete!')

# Encode labels: positive=1, negative=0
le = LabelEncoder()
df['sentiment'] = le.fit_transform(df['sentiment'])
print(f'\n🏷️  Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}')

df[['review', 'sentiment']].head(3)

## ✅ Step 5 — TF-IDF Vectorization
Converts text → numerical matrix (49,000 × 5,000)

In [ ]:
MAX_FEATURES = 5000  # Top 5000 most important words

tfidf = TfidfVectorizer(max_features=MAX_FEATURES)
X = tfidf.fit_transform(df['review']).toarray()
y = df['sentiment'].values

print(f'✅ Vectorization complete!')
print(f'📐 Feature Matrix Shape: {X.shape}  →  ({X.shape[0]} reviews × {X.shape[1]} features)')
print(f'🏷️  Labels Shape: {y.shape}')
print(f'\n📊 Class balance — Positive: {y.sum()} | Negative: {len(y) - y.sum()}')

## ✅ Step 6 — Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'✅ Data split complete!')
print(f'🏋️  Training samples : {len(X_train)}')
print(f'🧪 Testing  samples : {len(X_test)}')

# Convert to PyTorch tensors
train_ds = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32)
)
test_ds = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32)
)

# DataLoaders for batch processing
BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

print(f'\n📦 Batch size: {BATCH_SIZE}')
print(f'🔄 Training batches: {len(train_loader)}')

## ✅ Step 7 — Define RNN Model Architecture

In [ ]:
class SentimentRNN(nn.Module):
    """
    RNN model for binary sentiment classification.
    Architecture: RNN → Linear → Sigmoid
    """
    def __init__(self, input_size, hidden_size=128):
        super(SentimentRNN, self).__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        _, h_n = self.rnn(x)              # h_n: final hidden state
        out = self.fc(h_n.squeeze(0))     # Linear layer
        return self.sigmoid(out)          # Output probability 0-1


# Initialize model
model = SentimentRNN(input_size=MAX_FEATURES, hidden_size=128).to(device)

# Loss function & optimizer
criterion = nn.BCELoss()                              # Binary Cross Entropy
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer

# Count parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('✅ Model Architecture:')
print(model)
print(f'\n🔢 Total Trainable Parameters: {total_params:,}')

## ✅ Step 8 — Train the Model
> ⏱️ Expected time: ~5-10 minutes on CPU, ~1-2 minutes on GPU

In [ ]:
EPOCHS = 15
train_losses = []

print('🚀 Starting Training...')
print('=' * 55)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        # Move data to device (GPU if available)
        xb = xb.unsqueeze(1).to(device)   # Shape: [batch, 1, 5000]
        yb = yb.to(device)

        # Forward pass
        preds = model(xb).squeeze()
        loss = criterion(preds, yb)

        # Backward pass
        optimizer.zero_grad()   # Clear previous gradients
        loss.backward()         # Compute gradients
        optimizer.step()        # Update weights

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f'  Epoch [{epoch+1:02d}/{EPOCHS}]  |  Loss: {avg_loss:.4f}')

print('=' * 55)
print('✅ Training Complete!')

## ✅ Step 9 — Evaluate Model Accuracy

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.unsqueeze(1).to(device)
        yb = yb.to(device)

        preds = (model(xb).squeeze() > 0.5).float()  # Threshold at 0.5
        correct += (preds == yb).sum().item()
        total += yb.size(0)

accuracy = (correct / total) * 100
print(f'\n🎯 Test Results:')
print(f'   Correct Predictions : {correct}')
print(f'   Total Samples       : {total}')
print(f'   ✅ Final Accuracy   : {accuracy:.2f}%')

## ✅ Step 10 — Save Model & Vectorizer
> Both files **must be downloaded** and placed in the same folder as `app.py`

In [ ]:
# Save the trained RNN model weights
torch.save(model.state_dict(), 'rnn_model.pth')
print('✅ Model saved  →  rnn_model.pth')

# Save the TF-IDF vectorizer (CRITICAL: must match at inference time)
with open('tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
print('✅ Vectorizer saved  →  tfidf.pkl')

print('\n📁 Files to download and place next to app.py:')
print('   1. rnn_model.pth')
print('   2. tfidf.pkl')
print('\n💡 In Colab: Files panel (left sidebar) → right-click each file → Download')

## ✅ Step 11 — Quick Inference Test (Optional)
Test the model with a sample review before deploying

In [ ]:
def predict_sentiment(review_text):
    """Predict sentiment for a single review string"""
    cleaned = clean_text(review_text)
    vec = tfidf.transform([cleaned]).toarray()
    tensor = torch.tensor(vec, dtype=torch.float32).unsqueeze(1).to(device)

    model.eval()
    with torch.no_grad():
        prob = model(tensor).item()

    label = '😊 Positive' if prob >= 0.5 else '😞 Negative'
    confidence = prob if prob >= 0.5 else 1 - prob
    return label, confidence * 100


# Test with sample reviews
test_reviews = [
    "This movie was absolutely amazing! The acting was superb and the story kept me on the edge of my seat.",
    "Terrible film. Complete waste of time. The plot made no sense and acting was wooden.",
    "Decent movie, nothing special but enjoyable enough for a weekend watch."
]

print('🔍 Sample Predictions:\n')
for i, review in enumerate(test_reviews, 1):
    label, confidence = predict_sentiment(review)
    print(f'Review {i}: "{review[:60]}..."')
    print(f'  → Prediction: {label}  (Confidence: {confidence:.1f}%)\n')